# Notebook 02 — A proton beam and the dose it deposits

Notebook 01 followed single protons. A real irradiation is a **beam**: many
protons, each with a slightly different energy. Each one follows its own
random track. This notebook simulates 500 protons and looks at what they
deposit *together*:

1. the **depth-dose (Bragg) curve** and the **radial dose** around the
   beam axis;
2. **dose maps**: the pencil-beam kernel, and a broad beam built from it;
3. the protons' **energy before and after**;
4. the beam's **track- and dose-averaged LET**.

**Before you start.** Read notebook 01 first; this notebook uses the same
tables and assumes you know what a track and a stopping power are.

**Run time: about 14 minutes.** Almost all of it is the `run_beam` cell,
which simulates every collision of every proton, many thousands per
proton. To try things quickly, lower `N_PARTICLES` in the "Beam
parameters" cell, e.g. to 50 (about 1.5 minutes); the curves get noisier
but keep their shape. All figures are saved in `notebooks/output_02/`.

**How a beam is simulated.** `analysis.beam.run_beam` is a loop: draw an
energy, create a proton at the origin moving along $+z$, call
`TrackEngine.transport()`, repeat. All protons feed the **same scorers**,
so after the loop each scorer holds the energy deposited by the whole beam.

**Limitations to keep in mind** (see the main README):

- δ-electrons deposit their energy where they are created, so there is no
  δ-ray "halo" around the track;
- each track stops at 8 keV, the lowest tabulated energy, and the energy
  left at that point is not deposited;
- lateral scattering and depth-dose have not been validated against
  reference data.

In [ ]:
# Google Colab only: download the repository and install the package.
# On your own computer (after ./setup.sh) this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/ion-track-mc"):
        subprocess.run(["git", "clone", "-q", "https://github.com/juanmontiz/ion-track-mc",
                        "/content/ion-track-mc"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "/content/ion-track-mc"], check=True)
    os.chdir("/content/ion-track-mc/notebooks")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from pathlib import Path

from ion_track_mc import constants
from ion_track_mc.physics.channels import Channel
from ion_track_mc.physics.cross_section import (
    TabulatedCrossSectionSet,
    load_cross_section_table,
    load_mean_energy_table,
)
from ion_track_mc.physics.medium import WATER
from ion_track_mc.analysis.beam import run_beam
from ion_track_mc.analysis.visualization import (
    plot_bragg_curve,
    plot_radial_dose,
    save_figure,
    use_paper_style,
)
from ion_track_mc.scoring.dose import make_bragg_curve_scorer, RadialDoseScorer, VoxelDoseScorer

use_paper_style()

OUT_DIR = Path("output_02")  # every figure below is also saved here as PNG + PDF

## Load the cross sections and energy-loss tables

The same tables as in notebooks 01 and 03: cross sections (`TCS`) decide
how far the proton flies and which process happens; mean energies
(`Edepmeantot`, `Eemeantot`) decide how much energy each collision costs.
See `ion_track_mc/data/README_data_format.md` for what each file holds.

In [ ]:
DATA_DIR = Path("..") / "ion_track_mc" / "data" / "cross_sections"
FILES = {
    (1, Channel.IONIZATION): "H1+_+_H2O_ionization_TCS_CDW-EIS.dat",
    (1, Channel.EXCITATION): "H1+_+_H2O_excitation_TCS_Miller-Green.dat",
    (1, Channel.CAPTURE): "H1+_+_H2O_capture_TCS_CDW-EIS.dat",
    (1, Channel.ELASTIC): "H1+_+_H2O_elastic_TCS_Classical.dat",
    (0, Channel.IONIZATION): "H0_+_H2O_ionization_TCS_CDW-EIS.dat",
    (0, Channel.EXCITATION): "H0_+_H2O_excitation_TCS_Miller-Green.dat",
    (0, Channel.LOSS): "H0_+_H2O_loss_TCS_Miller-Green.dat",
    (0, Channel.ELASTIC): "H0_+_H2O_elastic_TCS_Classical.dat",
}
xs = TabulatedCrossSectionSet({k: load_cross_section_table(DATA_DIR / f) for k, f in FILES.items()})

EDEP_FILES = {
    (1, Channel.EXCITATION): "H1+_+_H2O_excitation_Edepmeantot_Miller-Green.dat",
    (1, Channel.IONIZATION): "H1+_+_H2O_ionization_Edepmeantot_CDW-EIS.dat",
    (1, Channel.CAPTURE):    "H1+_+_H2O_capture_Edepmeantot_CDW-EIS.dat",
    (0, Channel.EXCITATION): "H0_+_H2O_excitation_Edepmeantot_Miller-Green.dat",
    (0, Channel.IONIZATION): "H0_+_H2O_ionization_Edepmeantot_CDW-EIS.dat",
    (0, Channel.LOSS):       "H0_+_H2O_loss_Edepmeantot_constant.dat",
}
EEMEAN_FILES = {
    1: "H1+_+_H2O_ionization_Eemeantot_CDW-EIS.dat",
    0: "H0_+_H2O_ionization_Eemeantot_CDW-EIS.dat",
}
edep_tables = {k: load_mean_energy_table(DATA_DIR / f) for k, f in EDEP_FILES.items()}
eemean_tables = {k: load_mean_energy_table(DATA_DIR / f) for k, f in EEMEAN_FILES.items()}

## Beam parameters

- `N_PARTICLES = 500` protons.
- Mean energy 200 keV/u (for hydrogen, 200 keV). This is a low energy,
  chosen so that the whole track fits in about 2.5 µm and the run stays
  feasible on a laptop. Clinical proton beams have 70–250 **MeV** and
  ranges of centimetres; each track would then have millions of
  collisions.
- A **5% Gaussian energy spread** ($\sigma_E = 10$ keV), as a real
  accelerator beam has. Energies drawn outside the tabulated range would
  be redrawn; at 200 ± 10 keV that never happens.

In [ ]:
N_PARTICLES = 500
E_BEAM_KEV_U = 200.0
E_SIGMA_FRACTION = 0.05  # 5% Gaussian spread around E_beam

E_beam_eV = E_BEAM_KEV_U * 1.0e3 * constants.PROTON_NUCLEON_COUNT
E_sigma_eV = E_SIGMA_FRACTION * E_beam_eV
print(f"E_beam = {E_beam_eV:.1f} eV +/- {E_sigma_eV:.1f} eV, N = {N_PARTICLES}")

## Run the beam

Three **scorers** watch the same run. Each receives every energy deposit as
it happens (the "observer" pattern; see `ion_track_mc/scoring/README_scoring.md`):

| Scorer | Geometry | Gives |
|---|---|---|
| `bragg_scorer` | 60 depth bins of 50 nm, one huge transverse bin (±1 mm) | dose vs depth |
| `radial_scorer` | cylindrical shells around the $z$ axis, $r$ from 0.1 nm to 100 µm (log-spaced) | dose vs distance from the axis |
| `kernel_scorer` | 2 nm (lateral $x$) × 20 nm (depth $z$) voxels, summed over $y$ | 2D dose map |

`let_segment_length_nm=100.0` asks `run_beam` to also collect what it needs
for the dose-averaged LET (last section).

`seed=0` makes the run reproducible: the same seed always gives exactly the
same 500 tracks.

In [ ]:
bragg_scorer = make_bragg_curve_scorer(WATER, z_edges=np.linspace(0.0, 3000.0, 61),
                                        transverse_half_width_nm=1.0e6)
radial_scorer = RadialDoseScorer(WATER, r_edges=np.concatenate([[0.0], np.geomspace(0.1, 1.0e5, 29)]),
                                  length_nm=3000.0)
# Pencil-beam kernel for the dose maps further down: 2 nm (lateral x) x 20 nm (depth z)
# voxels, a single bin spanning all of y (so energy is summed over y).
KERNEL_DX, KERNEL_DZ = 2.0, 20.0
kernel_scorer = VoxelDoseScorer(WATER, x_edges=np.arange(-150.0, 150.0 + KERNEL_DX / 2, KERNEL_DX),
                                y_edges=np.array([-1.0e9, 1.0e9]),
                                z_edges=np.arange(0.0, 3000.0 + KERNEL_DZ / 2, KERNEL_DZ))

result = run_beam(WATER, xs, edep_tables=edep_tables, eemean_tables=eemean_tables,
                   n_particles=N_PARTICLES, E_beam_eV=E_beam_eV, E_sigma_eV=E_sigma_eV,
                   scorers=[bragg_scorer, radial_scorer, kernel_scorer], seed=0,
                   let_segment_length_nm=100.0)  # Delta for dose-averaged LET, see below

print(f"{N_PARTICLES} primaries transported, {result.n_rejected_out_of_range} energy resamples rejected")
print(f"total energy deposited: {sum(d.energy for d in result.engine.deposits):.1f} eV")
print(f"final charge state: {np.mean(result.q_final == 1)*100:.1f}% H+, {np.mean(result.q_final == 0)*100:.1f}% H0")

**Reading the printed numbers.**

- **Energy bookkeeping.** 500 protons × 200 keV = 100 MeV entered the
  water, and 96.1 MeV was deposited. The difference, about 4 MeV, is
  500 × 8 keV: each proton is no longer followed once it reaches the
  8 keV table floor, and that energy is not deposited.
- **Final charge state: 76% H⁰.** At the end of their tracks the
  projectiles are slow (8 keV). At that energy electron capture dominates,
  so most of them end up as neutral hydrogen atoms. At the entrance
  (200 keV) the large majority are bare protons.

## Bragg curve and radial dose

**Left: the depth-dose (Bragg) curve.** The dose in each 50 nm depth
slice, summed over all 500 protons.

**Right: the radial dose** $D(r)$, the dose in cylindrical shells at
distance $r$ from the beam axis, averaged over the first 3 µm of depth.
Both axes are logarithmic.

In this model a deposit can only be off the axis because the proton itself
has been deflected by elastic collisions. δ-electrons deposit where they
are created, on the proton's path.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
plot_bragg_curve(bragg_scorer, ax=axes[0], line_width=1.5)
plot_radial_dose(radial_scorer, ax=axes[1], line_width=1.5, marker_size=3.0)
fig.tight_layout()
save_figure(fig, OUT_DIR, "01_bragg_radial_dose")

**How to read the Bragg curve.**

- The dose rises from about 0.0016 Gy at the entrance to a **peak of about
  0.0021 Gy near 1.4 µm**, then falls to zero by about 2.4 µm. The fall-off
  is the end of the protons' **range**.
- The peak is only about 1.35× the entrance dose, much flatter than the
  3–4× of a clinical Bragg curve. This is real physics at this energy (see
  the comment after the broad-beam map below).
- The fall-off is not a vertical step. The 5% energy spread and the
  randomness of each track (**range straggling**) make different protons
  stop at slightly different depths.
- The *absolute* dose is tiny because the transverse bin is huge
  (2 mm × 2 mm): 500 protons spread over that area are a very low fluence.
  Only the *shape* of the curve is meaningful here. The broad-beam map
  below gives the dose at a realistic fluence.

**How to read the radial dose.**

- The dose is enormous at nanometre distances (up to $10^{9}$–$10^{10}$ Gy
  at $r < 1$ nm). Every proton deposits all its energy within a fraction of
  a nm of its own path, which is the essence of a "track" at the nanometre
  scale.
- It falls steeply with $r$, and is essentially zero beyond about 300 nm,
  the largest lateral distance any proton reaches.
- In reality δ-electrons would carry part of the energy further out,
  giving a $\sim 1/r^2$ tail. That tail is missing here because electrons
  are not transported.

## Dose maps: the pencil-beam kernel, and a broad beam by superposition

Every proton in `run_beam` enters at the same point and in the same
direction, so this beam is an infinitely thin **pencil beam**. Its dose map
in the depth–lateral plane is the **pencil-beam kernel**: how the energy
from one entry point spreads out as elastic scattering fans the protons
out. The `kernel_scorer` watched the same run as the other two scorers,
so the kernel costs no extra simulation.

The colour scale is logarithmic and shows energy per proton per voxel.
Black means no deposit.

In [ ]:
E_kernel = kernel_scorer.energy_grid_eV[:, 0, :]          # (n_x, n_z) eV, summed over y, all N_PARTICLES
x_k = kernel_scorer.x_edges
z_k = kernel_scorer.z_edges
z_c = 0.5 * (z_k[1:] + z_k[:-1])
print(f"kernel captured {kernel_scorer.total_deposited_eV() / result.total_delta_E_eV:.4f} of the beam's energy loss")

fig, ax = plt.subplots(figsize=(9, 3.2))
E_masked = np.ma.masked_equal(E_kernel / N_PARTICLES, 0.0)
im = ax.imshow(E_masked, origin="lower", extent=[z_k[0], z_k[-1], x_k[0], x_k[-1]], aspect="auto",
               cmap="magma", norm=LogNorm(vmin=E_masked.max() * 1e-4, vmax=E_masked.max()), interpolation="nearest")
ax.set_facecolor("black")
fig.colorbar(im, ax=ax, label="Energy per proton per voxel [eV] (log)")
ax.set_xlabel("Depth z [nm]")
ax.set_ylabel("Lateral x [nm]")
ax.set_title("Pencil-beam kernel (y-integrated)")
ax.set_ylim(-80, 80)
save_figure(fig, OUT_DIR, "02_pencil_beam_kernel")

**How to read the kernel.**

- At the entrance all protons share the same 2 nm wide voxel, so the energy
  per voxel is highest there. This is **not** the Bragg peak: deeper, the
  same energy is spread over a wider area. The map shows where the beam is
  *concentrated*, not how much energy each proton loses per nm.
- The beam fans out to about ±40–60 nm by the end of the range. Elastic
  deflections are small but add up, and they grow as the protons slow down.
- The isolated thin streaks are individual protons that were deflected
  early by a larger-than-usual angle.
- 99.9% of the beam's energy loss falls inside the map (printed above), so
  nothing important is cut off.

**Broad beam by superposition.** Water is homogeneous, so a beam of any shape is the sum of shifted copies of the pencil-beam kernel. For a beam uniform in $y$, with lateral profile $p(x)$ (peak 1) and peak fluence $\Phi$ [protons/nm²], the dose is the kernel convolved with the profile:

$$
D(x,z) \;=\; \frac{\Phi}{\rho}\,\sum_j p(x_j)\,\frac{E_{\text{bin}}(x - x_j,\,z)}{N\,\Delta z},
$$

where $E_{\text{bin}}$ is the energy the $N$ simulated protons deposited in each kernel voxel (summed over $y$), and $\Delta z$ is the depth bin. Below, $p(x)$ is a Gaussian with $\sigma = 500$ nm and $\Phi = 10^8$ protons/cm². Since this beam is much wider than the kernel (about ±40 nm at the end of the range), the centre-line dose must reduce to $\Phi\,S(z)/\rho$, fluence times stopping power over density; the cell checks that.

The vertical stripes are the statistical noise of each depth bin of the kernel, copied across the whole width by the convolution. More primaries smooth them out.

In [ ]:
SIGMA_BEAM_NM = 500.0
PHI_PER_NM2 = 1.0e8 / 1.0e14                      # 1e8 protons/cm^2 (1 cm^2 = 1e14 nm^2)
x_map = np.arange(-1500.0, 1500.0 + KERNEL_DX / 2, KERNEL_DX)
profile = np.exp(-0.5 * (x_map / SIGMA_BEAM_NM) ** 2)

rho_kg_per_nm3 = WATER.density_g_cm3 * 1.0e-3 / constants.CM3_TO_NM3
eV_per_nm2_per_proton = E_kernel / (N_PARTICLES * KERNEL_DX * KERNEL_DZ)    # y-integrated kernel density
energy_density = np.array([np.convolve(profile, eV_per_nm2_per_proton[:, iz], mode="same") * KERNEL_DX
                           for iz in range(len(z_c))]).T * PHI_PER_NM2      # eV/nm^3, shape (n_x, n_z)
dose_Gy = energy_density * constants.EV_TO_JOULE / rho_kg_per_nm3

# Check: centre line vs Phi * S(z) / rho, with S(z) the depth-dose per proton
S_of_z = E_kernel.sum(axis=0) / (N_PARTICLES * KERNEL_DZ)                 # eV/nm
centre_expected = PHI_PER_NM2 * S_of_z * constants.EV_TO_JOULE / rho_kg_per_nm3
i0 = len(x_map) // 2
print(f"peak dose {dose_Gy.max():.1f} Gy at z = {z_c[np.argmax(dose_Gy[i0])]:.0f} nm; "
      f"centre line vs Phi*S/rho: max rel. difference {np.max(np.abs(dose_Gy[i0] - centre_expected)) / centre_expected.max():.4f}")

fig = plt.figure(figsize=(9, 6))
gs = fig.add_gridspec(2, 2, width_ratios=[1, 0.03], height_ratios=[2, 1], hspace=0.35, wspace=0.03)
ax, cax = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1])
ax2 = fig.add_subplot(gs[1, 0], sharex=ax)
im = ax.imshow(dose_Gy, origin="lower", extent=[z_k[0], z_k[-1], x_map[0], x_map[-1]], aspect="auto",
               cmap="magma", interpolation="bilinear")
fig.colorbar(im, cax=cax, label="Dose [Gy]")
ax.set_ylabel("Lateral x [nm]")
ax.set_title(f"Broad beam: Gaussian sigma = {SIGMA_BEAM_NM:.0f} nm, peak fluence 1e8 cm$^{{-2}}$")
plt.setp(ax.get_xticklabels(), visible=False)
for x_line in (0.0, SIGMA_BEAM_NM, 2.0 * SIGMA_BEAM_NM):
    i = np.argmin(np.abs(x_map - x_line))
    ax2.plot(z_c, dose_Gy[i], label=f"x = {x_line:.0f} nm")
    ax.axhline(x_line, color="w", lw=0.6, ls=":")
ax2.set_xlabel("Depth z [nm]")
ax2.set_ylabel("Dose [Gy]")
ax2.set_title("Depth-dose at fixed lateral position")
ax2.legend(fontsize=8)
save_figure(fig, OUT_DIR, "03_broad_beam_dose_map")

**How to read the broad-beam map.** The Bragg peak is now visible along
depth: 17.0 Gy at $z \approx 1.33$ µm for a peak fluence of
$10^8$ protons/cm². It is only about 1.35× the entrance dose, much flatter
than a clinical Bragg curve (~3–4×). That is real physics at this energy,
not an artifact. The stopping power of water peaks near 80 keV (NIST
PSTAR; about 70 keV/u in this model), and a 200 keV proton starts not far
above that maximum, so its rate of energy loss changes little before the
end. Clinical beams start far above the maximum, where the stopping power
is about 100 times lower than at the maximum; the clinical
peak-to-entrance ratio of 3–4 is smaller than that because range
straggling and the sum over many protons smear the peak out.

The centre-line check printed above agrees with $\Phi\,S(z)/\rho$ to
0.14%, confirming the superposition.

Caveats, as elsewhere: tracks stop at the 8 keV table floor (residual
energy not deposited), electrons deposit locally (no δ-ray halo), and the
lateral scattering and depth-dose are not yet validated against reference
data.

## Energy of each proton, before and after

A histogram of each proton's energy at the start (blue) and when it stopped
being followed (orange).

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.hist(result.E_start_eV, bins=25, alpha=0.6, label="start")
ax.hist(result.E_final_eV, bins=25, alpha=0.6, label="final (at e_cut / range exit)")
ax.set_xlabel("Energy [eV]")
ax.set_ylabel("Particle count")
ax.legend()
save_figure(fig, OUT_DIR, "04_energy_distributions")
plt.show()

**How to read this histogram.** The blue histogram is the Gaussian
energy spread of the beam, centred at 200 keV with $\sigma = 10$ keV.

The orange ("final") histogram is almost invisible, and that is the
point: **every proton ends at the same energy**, just below the 8 keV table
floor where tracking stops. All 500 final energies lie within a few eV of
8 keV, so the histogram's 25 bins are all crowded into that tiny interval
at the left edge of the plot, too thin to see. The beam's energy spread is
not carried to the end of the track; what it changes is *how far* each
proton gets before reaching 8 keV, i.e. its range.

## Beam-averaged LET

LET (linear energy transfer) is the energy the proton loses per unit path
length. It changes along every track: low at the entrance, high near the
end. Two averages summarise it (see `ion_track_mc/scoring/README_scoring.md` §3):

- **Track-averaged LET**, $\bar L_T = \sum \Delta E / \sum s$: the total
  energy lost by all protons divided by the total path they flew. Every nm
  of track counts equally.
- **Dose-averaged LET**, $\bar L_D$: each piece of track is weighted by
  the energy it deposits, so the high-LET regions (where most of the dose
  is) count more. $\bar L_D$ is the average usually related to biological
  effect, and it is always $\ge \bar L_T$.

$\bar L_D$ is computed over track segments of fixed length $\Delta$, here
100 nm (the `let_segment_length_nm` passed to `run_beam`). A fixed length is
needed because LET measured over a single collision step does not converge
to a stable value. The result therefore depends on $\Delta$; notebook 01 §9
shows how much. Without $\Delta$, `result.dose_averaged_LET()` raises an
error.

`run_beam` adds up the necessary sums track by track while the beam runs,
so the result is a true average over all 500 protons.

LET in eV/nm is numerically equal to keV/µm, the unit usually quoted.

In [ ]:
print(f"beam track-averaged LET = {result.track_averaged_LET():.3f} eV/nm (= keV/um)")
print(f"beam dose-averaged LET  = {result.dose_averaged_LET():.3f} eV/nm (= keV/um), "
      f"Delta = {result.let_segment_length_nm:g} nm")

**Reading the result.** $\bar L_T \approx 88.6$ keV/µm and
$\bar L_D \approx 91.3$ keV/µm at $\Delta = 100$ nm. The two are close
because, at 200 keV, the stopping power changes little along most of the
track: the energy is fairly evenly spread, as the flat Bragg curve above
already showed. For a clinical beam, which starts far above the stopping-power
maximum, $\bar L_D$ would exceed $\bar L_T$ by much more.

These values are typical of the **end of a proton's range** (the Bragg
peak region of a clinical beam), where LET is highest. Clinical protons at
the entrance have LET of only about 0.5 keV/µm.

## Stopping power

Stopping power (Monte Carlo vs. analytic self-consistency, NIST PSTAR, the charge-equilibrium blend, and experimental data) is covered in **notebook 01, sections 4–7**. It is not repeated here, so this notebook stays focused on the beam and the dose it deposits.